# Week 6 (starter): Advanced Retrieval, Transform and Measure

Extends Week 5. The transformation shown here is pseudo-relevance feedback (PRF), which is fully local and needs no key; HyDE is guarded behind a key. Cells marked **TODO (you)** are yours. Dependency: `sentence-transformers`. For HyDE: `pip install openai` and a Gemini key.

### Part 1: Transform and re-rank


In [35]:
import os
import re
import pathlib
import numpy as np

from collections import Counter
from pathlib import Path
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer, util


# --------------------------------------------------
# Embedding model
# --------------------------------------------------

os.environ['HF_HOME'] = str(
    (pathlib.Path('.') / '.hf_cache').resolve()
)

embedder = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)


# --------------------------------------------------
# Load Spring HTML documents
# --------------------------------------------------

documents = []

for path in Path("documents").glob("*.html"):
    html = path.read_text(encoding="utf-8")

    soup = BeautifulSoup(html, "html.parser")

    # Remove content we do not want embedded
    for tag in soup(["script", "style", "nav"]):
        tag.decompose()

    text = soup.get_text(" ", strip=True)

    documents.append({
        "title": path.stem,
        "text": text
    })

print(f"Loaded {len(documents)} documents")


# --------------------------------------------------
# Chunking function
# Week 5 strategy: 320 characters / 40 overlap
# --------------------------------------------------

def chunk_text(text, chunk_size=320, overlap=40):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------
# Create labeled corpus
#
# Each chunk keeps the title of the HTML document
# that it came from as its topic.
#
# Example:
# ("spring-security", "chunk text...")
# --------------------------------------------------

corpus = []

for doc in documents:
    topic = doc["title"]

    chunks = chunk_text(
        doc["text"],
        chunk_size=320,
        overlap=40
    )

    for chunk in chunks:
        corpus.append((topic, chunk))


# --------------------------------------------------
# Match professor's original corpus structure
# --------------------------------------------------

topics = [c[0] for c in corpus]
docs = [c[1] for c in corpus]


# --------------------------------------------------
# Embed chunks
# --------------------------------------------------

doc_emb = embedder.encode(
    docs,
    convert_to_tensor=True,
    normalize_embeddings=True
)


# --------------------------------------------------
# Retrieval settings
# --------------------------------------------------

K = 4

# --------------------------------------------------
# Sanity checks
# --------------------------------------------------

print("Number of chunks:", len(docs))
print("Number of topic labels:", len(topics))
print("Embedding shape:", doc_emb.shape)

print("\nUnique topics:")
print(set(topics))

assert len(docs) == len(topics)
assert len(docs) == doc_emb.shape[0]

Loaded 9 documents
Number of chunks: 581
Number of topic labels: 581
Embedding shape: torch.Size([581, 384])

Unique topics:
{'Spring-Batch', 'Bean-Annotation', 'Spring-JPA-Repository', 'Spring-Data', 'Spring-Data-JPA', 'Spring-Security-OAuth2', 'Spring-Authorization', 'Spring-Dependencies', 'Spring-Security'}


In [36]:
queries = [
    ("Bean-Annotation",
     "What does the @Bean annotation do?"),

    ("Spring-Authorization",
     "What do GrantedAuthority objects represent in Spring Security?"),

    ("Spring-Batch",
     "Is Spring Batch a scheduling framework?"),

    ("Spring-Data-JPA",
     "What does Spring Data JPA provide?"),

    ("Spring-JPA-Repository",
     "What is the central interface in the Spring Data repository abstraction?"),

    ("Spring-Dependencies",
     "What are the two main types of dependency injection in Spring?"),

    ("Spring-JPA-Repository",
     "What functionality do CrudRepository and ListCrudRepository provide?"),

    ("Spring-Security-OAuth2",
     "What is the base property prefix for OAuth Client registration properties?"),

    ("Spring-Security",
     "What does SecurityFilterChain determine in Spring Security?"),

    ("Spring-Security",
     "What is the purpose of DelegatingFilterProxy in Spring Security?")
]

K = 4


In [37]:
STOP=set('a an the of to in on for and or is are how do does what so that with by as it why'.split())
def retrieve_idx(q,k=K):
    qe=embedder.encode([q],convert_to_tensor=True,normalize_embeddings=True)
    sims=util.cos_sim(qe,doc_emb).cpu().numpy()[0]; return list(np.argsort(sims)[::-1][:k])
def prf_expand(q, feedback_k=3, add=4):
    top=retrieve_idx(q,feedback_k); words=re.findall(r'[a-zA-Z]+',' '.join(docs[i] for i in top).lower())
    cand=[w for w in words if w not in STOP and len(w)>3 and w not in q.lower()]
    return q+' '+' '.join(w for w,_ in Counter(cand).most_common(add))
def precision(idxs,topic): return sum(1 for i in idxs if topics[i]==topic)/K

print('per-query precision@4, baseline -> PRF:')
improved=regressed=0
for topic,q in queries:
    print(f"topic = {topic}")
    b=precision(retrieve_idx(q),topic); a=precision(retrieve_idx(prf_expand(q)),topic)
    flag='IMPROVED' if a>b else ('REGRESSED' if a<b else 'same')
    improved+=a>b; regressed+=a<b
    print(f'  [{topic:5s}] {b:.2f} -> {a:.2f}  {flag}   {q!r}')
print(f'\nimproved {improved}, regressed {regressed}')

per-query precision@4, baseline -> PRF:
topic = Bean-Annotation
  [Bean-Annotation] 0.75 -> 1.00  IMPROVED   'What does the @Bean annotation do?'
topic = Spring-Authorization
  [Spring-Authorization] 1.00 -> 1.00  same   'What do GrantedAuthority objects represent in Spring Security?'
topic = Spring-Batch
  [Spring-Batch] 1.00 -> 1.00  same   'Is Spring Batch a scheduling framework?'
topic = Spring-Data-JPA
  [Spring-Data-JPA] 0.50 -> 0.50  same   'What does Spring Data JPA provide?'
topic = Spring-JPA-Repository
  [Spring-JPA-Repository] 0.50 -> 0.50  same   'What is the central interface in the Spring Data repository abstraction?'
topic = Spring-Dependencies
  [Spring-Dependencies] 1.00 -> 1.00  same   'What are the two main types of dependency injection in Spring?'
topic = Spring-JPA-Repository
  [Spring-JPA-Repository] 0.50 -> 0.50  same   'What functionality do CrudRepository and ListCrudRepository provide?'
topic = Spring-Security-OAuth2
  [Spring-Security-OAuth2] 1.00 -> 1.00  s

In [38]:
from sentence_transformers import CrossEncoder

# Load cross-encoder reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def retrieve_candidates(q, k=10):
    """
    Retrieve a larger candidate set using embedding similarity.
    """
    q_emb = embedder.encode(
        q,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    scores = util.cos_sim(q_emb, doc_emb)[0]

    return scores.topk(k).indices.tolist()


def rerank(q, candidate_idxs, top_k=4):
    """
    Rerank candidate chunks using a CrossEncoder.
    """

    pairs = [
        (q, docs[i])
        for i in candidate_idxs
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidate_idxs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [idx for idx, score in ranked[:top_k]]


def advanced_retrieve(q, candidate_k=10, final_k=4):
    """
    Advanced Week 6 retrieval pipeline:

    1. Expand original query using PRF
    2. Retrieve top candidate chunks using expanded query
    3. Rerank candidates against the original query
    4. Return final top chunks
    """

    # Query transformation
    expanded_q = prf_expand(q)

    # Retrieve larger candidate pool
    candidate_idxs = retrieve_candidates(
        expanded_q,
        k=candidate_k
    )

    # Rerank candidates using original query
    final_idxs = rerank(
        q,
        candidate_idxs,
        top_k=final_k
    )

    return final_idxs


# Compare baseline retrieval against PRF + reranking
print("per-query precision@4, baseline -> PRF + rerank:")

improved = 0
regressed = 0

for topic, q in queries:

    # Normal Week 5-style retrieval
    baseline_idxs = retrieve_idx(q)
    baseline_precision = precision(baseline_idxs, topic)

    # Week 6 advanced retrieval
    advanced_idxs = advanced_retrieve(q)
    advanced_precision = precision(advanced_idxs, topic)

    delta = advanced_precision - baseline_precision

    if delta > 0:
        flag = "IMPROVED"
        improved += 1
    elif delta < 0:
        flag = "REGRESSED"
        regressed += 1
    else:
        flag = "same"

    print(
        f"[{topic}] "
        f"{baseline_precision:.2f} -> {advanced_precision:.2f} "
        f"(delta {delta:+.2f}) "
        f"{flag} "
        f"{q!r}"
    )


print()
print(f"improved {improved}, regressed {regressed}")

per-query precision@4, baseline -> PRF + rerank:
[Bean-Annotation] 0.75 -> 1.00 (delta +0.25) IMPROVED 'What does the @Bean annotation do?'
[Spring-Authorization] 1.00 -> 1.00 (delta +0.00) same 'What do GrantedAuthority objects represent in Spring Security?'
[Spring-Batch] 1.00 -> 1.00 (delta +0.00) same 'Is Spring Batch a scheduling framework?'
[Spring-Data-JPA] 0.50 -> 0.50 (delta +0.00) same 'What does Spring Data JPA provide?'
[Spring-JPA-Repository] 0.50 -> 0.50 (delta +0.00) same 'What is the central interface in the Spring Data repository abstraction?'
[Spring-Dependencies] 1.00 -> 1.00 (delta +0.00) same 'What are the two main types of dependency injection in Spring?'
[Spring-JPA-Repository] 0.50 -> 0.50 (delta +0.00) same 'What functionality do CrudRepository and ListCrudRepository provide?'
[Spring-Security-OAuth2] 1.00 -> 1.00 (delta +0.00) same 'What is the base property prefix for OAuth Client registration properties?'
[Spring-Security] 1.00 -> 1.00 (delta +0.00) same 'Wh

## Part 2: Measure the Effect

I used the same queries from Week 5 that analyzed the Java Spring documentation. I first developed a baseline by running the Week 5 queries and evaluating the top 4 retrieved chunks using Precision@4. For this evaluation, a retrieved chunk was considered relevant if it came from the expected Spring documentation source for that query. Therefore, Precision@4 measured the proportion of the top four retrieved chunks that came from the expected document. I then used PRF query transformation to potentially enhance the retrieval process. After PRF, I retrieved a larger set of candidate chunks and used a CrossEncoder to rerank those chunks based on their relevance to the original query. Finally, I measured the retrieval quality of the advanced PRF + reranking pipeline using the same Precision@4 metric and compared the results against the baseline.

| Query | Baseline P@4 | PRF P@4 | PRF + Re-rank P@4 | Overall Delta |
|---|---:|---:|---:|---:|
| What does the @Bean annotation do? | 0.75 | 1.00 | 1.00 | +0.25 |
| What do GrantedAuthority objects represent in Spring Security? | 1.00 | 1.00 | 1.00 | 0.00 |
| Is Spring Batch a scheduling framework? | 1.00 | 1.00 | 1.00 | 0.00 |
| What does Spring Data JPA provide? | 0.50 | 0.50 | 0.50 | 0.00 |
| What is the central interface in the Spring Data repository abstraction? | 0.50 | 0.50 | 0.50 | 0.00 |
| What are the two main types of dependency injection in Spring? | 1.00 | 1.00 | 1.00 | 0.00 |
| What functionality do CrudRepository and ListCrudRepository provide? | 0.50 | 0.50 | 0.50 | 0.00 |
| What is the base property prefix for OAuth Client registration properties? | 1.00 | 1.00 | 1.00 | 0.00 |
| What does SecurityFilterChain determine in Spring Security? | 1.00 | 1.00 | 1.00 | 0.00 |
| What is the purpose of DelegatingFilterProxy in Spring Security? | 1.00 | 1.00 | 1.00 | 0.00 |
| **Mean** | **0.825** | **0.850** | **0.850** | **+0.025** |

The only query that saw an improvement was `What does the @Bean annotation do?`, which had a delta of +0.25, improving from 0.75 to 1.00. All other queries saw no change across the baseline, PRF, and PRF + reranking approaches. Overall, mean Precision@4 improved from 0.825 for the baseline to 0.850 after PRF. The reranking step maintained the mean Precision@4 of 0.850, but did not produce any additional improvement for this query set.



## Part 3: Failure Analysis

None of the 10 queries regressed after applying PRF and re-ranking. I had 1 improvement, 9 unchanged queries, and 0 regressions. Little improvement was seen from the baseline to PRF because the initial queries were already very strong and contained specific technical terms that were highly relevant to the expected documentation. This likely explains both the minimal improvement and the lack of regressions from PRF and PRF + re-ranking. Since the original queries already provided strong retrieval signals, transforming them did not substantially change the retrieved results.